In [ ]:
import pandas as pd
import requests
import time
import logging
import finnhub
import json

In [ ]:
from Important_constants import Financial_Constants as fc

### Getting the data

In [ ]:
logging.basicConfig(filename = './Logs/resultLogsAlternative.log', level = logging.INFO, format = "%(levelname)s - %(message)s", filemode='a')

In [ ]:
API_KEY = fc.FIN_API_KEY
EXCHANGE_RATE_KEY = fc.EXCHANGERATE_API_KEY

# Preparing the currencies Dictionary

In [ ]:
currencies_url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_KEY}/latest/USD'
response = requests.get(currencies_url, timeout = 8)
data = response.json()

In [ ]:
print(data.get("conversion_rates"))

In [ ]:
currencies_dict = data.get("conversion_rates")
print(type(currencies_dict))

In [ ]:
print(currencies_dict["ARS"])

### Alternative way using European centrobank API

In [ ]:
link_to_table = "https://data-api.ecb.europa.eu/service/data/EXR/D..EUR.SP00.A?lastNObservations=1&format=csvdata"
df = pd.read_csv(link_to_table)

In [ ]:
print(df)

# Reading the data

In [ ]:
tickers = pd.read_csv('../Data/All_Stocks.csv')
to_clean = pd.DataFrame(tickers)
OUTPUT_PATH = '../Data/EPS_Bigger_Than_PE_alternative.csv'
CURRENT_DELAY = 2.0

In [ ]:
url = f"https://finnhub.io/api/v1/stock/profile2?symbol=KB&token={API_KEY}"
response =  requests.get(url, timeout = 8)
print(response.json().get("estimateCurrency"))

### 'Cleaning' Data

In [ ]:
pattern = r'\^'
rows_to_drop = to_clean[to_clean['Symbol'].str.contains(pattern, na = False)].index
cleaned = to_clean.drop(rows_to_drop)
cleaned['Symbol'] = cleaned['Symbol'].str.replace(r'[\\/]', '-', regex = True)
cleaned['Symbol'] = cleaned['Symbol'].str.strip()
cleaned = cleaned['Symbol'].tolist()

## Writing functions that checks whether the stock TTM P/E is lower than EPS

In [ ]:
def check_the_stock(ticker : str):
    if ticker is None:
        return None
    url_path = f'https://finnhub.io/api/v1/stock/metric?symbol={ticker}&metric=all&token={API_KEY}'
    try:
        response = requests.get(url_path, timeout=8)
        if response.status_code == 200:
            allData = response.json()
            metrics = allData.get('metric', {})
            currentPE = metrics.get('peTTM')
            EPS = metrics.get('epsTTM')
            # currentPE = round(currentPE, 2)
            # EPS = round(EPS, 2)
            if EPS is None or EPS <= 0:
                logging.warning(f"Ticker {ticker} have negative EPS")
                return None
            elif EPS > currentPE:
                currentPE = round(currentPE, 2)
                EPS = round(EPS, 2)
                ps = round(metrics.get('psTTM'), 2)
                pb = round(metrics.get('pb'), 2)
                logging.critical(f"MATCH!!!!!!!!!!!!!!!!!!!!!!!!, {ticker}, P/E {currentPE} < EPS {EPS}, P/S : {ps}, P/B : {pb}")
                return {
                    'Ticker' : ticker,
                    'Trailing P/E' : currentPE,
                    'TTM_EPS' : EPS,
                    'P/S' : ps,
                    'P/B' : pb
                }
            else:
                logging.info(f"Ticker {ticker} P/E is bigger than EPS, {currentPE} > {EPS}")
                return None
    except Exception as e:
        print(e)
        logging.error(e)

In [ ]:
def check_the_stock_with_session(ticker : str, uSession = None):
    sessionCreated = False
    if uSession is None:
        uSession = requests.Session()
        sessionCreated = True
    if ticker is None:
        return None
    url_path = f'https://finnhub.io/api/v1/stock/metric?symbol={ticker}&metric=all&token={API_KEY}'
    try:
        response = uSession.get(url_path, timeout=8)
        if response.status_code == 200:
            allData = response.json()
            metrics = allData.get('metric', {})
            currentPE = metrics.get('peTTM')
            EPS = metrics.get('epsTTM')
            # currentPE = round(currentPE, 2)
            # EPS = round(EPS, 2)
            if EPS is None or EPS <= 0:
                logging.warning(f"Ticker {ticker} have negative EPS")
                return None
            elif currentPE is None or currentPE <= 0:
                logging.warning(f"Ticker {ticker} have negative PE")
                return None
            elif EPS > currentPE:
                currentPE = round(currentPE, 2)
                EPS = round(EPS, 2)
                ps = round(metrics.get('psTTM'), 2)
                pb = round(metrics.get('pb'), 2)
                logging.critical(f"MATCH!!!!!!!!!!!!!!!!!!!!!!!!, {ticker}, P/E {currentPE} < EPS {EPS}, P/S : {ps}, P/B : {pb}")
                return {
                    'Ticker' : ticker,
                    'Trailing P/E' : currentPE,
                    'TTM_EPS' : EPS,
                    'P/S' : ps,
                    'P/B' : pb
                }
            else:
                logging.info(f"Ticker {ticker} P/E is bigger than EPS, {currentPE} > {EPS}")
                return None
    except Exception as e:
        print(f"Something Wrong has happened with {ticker}")
        print(e)
        logging.error(e)
    finally:
        if sessionCreated:
            uSession.close()

In [ ]:
print(check_the_stock('CHTR'))

# Testing

In [ ]:
results = []

In [ ]:
# For testing purposes:
#cleaned = cleaned[:100]

In [ ]:
for index_of_ticker, ticker in enumerate(cleaned, 1):
    ticker_data = check_the_stock(ticker)
    if ticker_data is not None:
        print(ticker_data)
        results.append(ticker_data)
    time.sleep(CURRENT_DELAY)
    if index_of_ticker % 100 == 0:
        print(f"Stock number {index_of_ticker} finished")

In [ ]:
currSession = requests.Session()
try:
    for index_of_ticker, ticker in enumerate(cleaned, 1):
        ticker_data = check_the_stock_with_session(ticker, currSession)
        if ticker_data is not None:
            print(ticker_data)
            results.append(ticker_data)
        time.sleep(CURRENT_DELAY)
        if index_of_ticker % 100 == 0:
            print(f"Stock number {index_of_ticker} finished")
except Exception as e:
    print(e)
finally:
    currSession.close()

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_PATH, index = False)
print('Finished')

In [ ]:
url_path = f'https://finnhub.io/api/v1/stock/metric?symbol=CHTR&metric=all&token={API_KEY}'
resp = requests.get(url_path)
if resp.status_code == 200:
    allData = resp.json()
    print(allData)